In [ ]:
import sqlite3
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

from churn_analysis.config import DB_PATH, DB_TABLE_NAME

In [ ]:
SCHEMA_MAP = {
    "customer_id": "string",
    "gender": "string",
    "senior_citizen": "boolean",
    "partner": "boolean",
    "dependents": "boolean",
    "tenure": "Int64",
    "phone_service": "boolean",
    "multiple_lines": "string",
    "internet_services": "string",
    "online_security": "boolean",
    "online_backup": "boolean",
    "device_protection": "boolean",
    "tech_support": "boolean",
    "streaming_tv": "boolean",
    "streaming_movies": "boolean",
    "contract": "string",
    "paper_less_billing": "boolean",
    "payment_method": "string",
    "monthly_charges": "Float64",
    "total_charges": "Float64",
    "churn": "boolean",
}


def execute_query(query: str) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        result = pd.read_sql_query(sql=query, con=conn, dtype=SCHEMA_MAP)
    return result

In [ ]:
source_df = execute_query(f"SELECT * FROM {DB_TABLE_NAME}")
NUMB_ROWS = source_df.index.size
source_df = source_df.drop(columns="customer_id")
# CHURN COUNTS
source_df["churn"].value_counts()

In [ ]:
# GET THE CORRELATION AND FREQUENCY OF THE TARGET VARIABLE WITH EACH FEATURE
def get_encoded_feat_corr_freq(data_dummed: pd.DataFrame, target: str) -> pd.DataFrame:
    data_dummed = deepcopy(data_dummed)
    encoded_feat_frequency = []
    target_corr = data_dummed.corr(method="pearson").round(2)[target]
    for col_name in target_corr.index:
        encoded_feat_frequency.append(data_dummed[col_name].value_counts()[True])

    encoded_feat_corr = (
        pd.DataFrame(
            {
                "encoded_features": target_corr.index,
                "correlation_to_churn": target_corr.values,
                "encoded_features_true_values_frequency": encoded_feat_frequency,
            }
        )
        .sort_values("correlation_to_churn", ascending=False)
        .reset_index(drop=True)
    )
    return encoded_feat_corr

In [ ]:
str_col_names = source_df.select_dtypes(include="string").columns
df_dummed = pd.get_dummies(data=source_df, columns=str_col_names)
feature_correlation_to_churn = get_encoded_feat_corr_freq(
    data_dummed=df_dummed.select_dtypes(include="boolean"), target="churn"
)
feature_correlation_to_churn

In [ ]:
# GET NUMERIC HEATMAP FOR CORRELATION ANALYSIS
df_dummed = pd.get_dummies(data=source_df, columns=str_col_names, drop_first=True)
numeric_df_dummed = df_dummed.select_dtypes(include="number")
numeric_corr = numeric_df_dummed.corr(method="pearson").abs().round(2)
fig, ax = plt.subplots(figsize=(10, 10))
mask_numeric = np.triu(np.ones_like(numeric_corr, dtype=bool), k=1)
sns.heatmap(data=numeric_corr, annot=True, cmap="Blues", mask=mask_numeric, ax=ax)
plt.show()
plt.close(fig=fig)

In [ ]:
# GET NUMERIC HEATMAP FOR CORRELATION ANALYSIS
boolean_df_dummed = df_dummed.select_dtypes(include="boolean")
boolean_corr = boolean_df_dummed.corr(method="pearson").abs().round(2)
fig, ax = plt.subplots(figsize=(25, 25))
mask_boolean = np.triu(np.ones_like(boolean_corr, dtype=bool), k=1)
sns.heatmap(data=boolean_corr, annot=True, cmap="Blues", mask=mask_boolean, ax=ax)
plt.show()
plt.close(fig=fig)

In [ ]:
# REMOVE FEATURE BASED ON ITS CORRELATION TO OTHER VARIABLES
def get_cols_to_drop(correlations: pd.DataFrame, mask: npt.NDArray[np.float64]):
    masked_corr = correlations.where(cond=mask)
    cols_to_drop = [
        col_name
        for col_name in correlations.columns
        if np.any(masked_corr[col_name] > 0.8)
    ]
    return cols_to_drop


num_col_to_drop = get_cols_to_drop(correlations=numeric_corr, mask=mask_numeric)
boolean_col_to_drop = get_cols_to_drop(correlations=boolean_corr, mask=mask_boolean)
df_feature_engineering = df_dummed.drop(
    columns=[*num_col_to_drop, *boolean_col_to_drop]
)
print([*num_col_to_drop, *boolean_col_to_drop])
df_feature_engineering.head()

In [ ]:
# SCALE NUMERIC VARIABLES
def minmax_scale(dataframe: pd.DataFrame):
    dataframe = deepcopy(dataframe)
    num_df_feat_eng_cols = dataframe.select_dtypes(include="number").columns
    scaler = MinMaxScaler()
    dataframe[num_df_feat_eng_cols] = scaler.fit_transform(
        dataframe[num_df_feat_eng_cols]
    )
    new_cols = {col: f"scaled_{col}" for col in num_df_feat_eng_cols}
    dataframe = dataframe.rename(columns=new_cols)
    return dataframe


scaled_df = minmax_scale(dataframe=df_feature_engineering)
scaled_df.head()


In [ ]:
# CONVERT BOOLEAN VARIABLES TO INTEGERS
boolean_cols = scaled_df.select_dtypes(include="boolean").columns
scaled_df[boolean_cols] = scaled_df[boolean_cols].astype(dtype="Int64")
scaled_df.head()

In [ ]:
_